# Modelado: Baseline + XGBoost + LightGBM

| Modelo | Desbalance | Notas |
|---|---|---|
| Logistic Regression | `class_weight='balanced'` | Baseline interpretable |
| XGBoost | `scale_pos_weight` | ~14.6× peso a clase positiva |
| LightGBM | `is_unbalance=True` | Ajuste automático |

**Métricas principales:** ROC-AUC (ranking), PR-AUC (mejor para desbalance), F1 en umbral óptimo.

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.append(str(ROOT))

from src.models.train_06 import (
    load_data, get_models, cross_validate_models,
    evaluate_on_test, plot_roc_pr, plot_shap,
    best_f1_threshold,
    FEATURE_COLS, MODELS_DIR,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
sns.set_theme(style='whitegrid')
%matplotlib inline

RANDOM_STATE = 42

## 1. Cargar datos

In [ ]:
X, y = load_data()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'scale_pos_weight: {pos_weight:.2f}')

## 2. Validación cruzada (5 folds)

In [ ]:
models = get_models(pos_weight)
cv_results = cross_validate_models(X_train, y_train, models)
cv_results

In [ ]:
# Visualización de resultados CV
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, label in zip(axes, ['auc_mean', 'pr_auc_mean'], ['ROC-AUC', 'PR-AUC']):
    err = cv_results[metric.replace('mean', 'std')]
    ax.barh(cv_results['model'], cv_results[metric], xerr=err,
            color=['#F44336', '#4CAF50', '#2196F3'][:len(cv_results)],
            edgecolor='white', capsize=4)
    ax.set_title(f'{label} — 5-fold CV')
    ax.set_xlabel(label)
    for i, (v, e) in enumerate(zip(cv_results[metric], err)):
        ax.text(v + e + 0.001, i, f'{v:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 3. Entrenamiento final y evaluación en test

In [ ]:
final_models = get_models(pos_weight)
for name, model in final_models.items():
    model.fit(X_train, y_train)
    print(f'{name} ✓')

In [ ]:
test_results = [evaluate_on_test(n, m, X_test, y_test) for n, m in final_models.items()]

## 4. Curvas ROC y Precision-Recall

In [ ]:
plot_roc_pr(test_results, y_test)

## 5. SHAP — Interpretabilidad del mejor modelo

In [ ]:
best_name = cv_results.iloc[0]['model']
best_model = final_models[best_name]
print(f'Mejor modelo: {best_name}')

from sklearn.pipeline import Pipeline
if not isinstance(best_model, Pipeline):
    plot_shap(best_model, X_test)

## 6. Análisis de umbral óptimo

In [ ]:
from sklearn.metrics import precision_recall_curve, f1_score

best_result = next(r for r in test_results if r['name'] == best_name)
proba = best_result['proba']

prec, rec, thresholds = precision_recall_curve(y_test, proba)
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(thresholds, prec[:-1], color='#2196F3', label='Precision')
axes[0].plot(thresholds, rec[:-1], color='#4CAF50', label='Recall')
axes[0].plot(thresholds, f1s, color='#F44336', label='F1', lw=2)
axes[0].axvline(thresholds[np.argmax(f1s)], color='black', linestyle='--',
                label=f'Óptimo = {thresholds[np.argmax(f1s)]:.2f}')
axes[0].set_title(f'Precision / Recall / F1 vs umbral — {best_name}')
axes[0].set_xlabel('Umbral')
axes[0].legend()

# Distribución de scores por clase
axes[1].hist(proba[y_test == 0], bins=60, alpha=0.6, color='#4CAF50',
             label='Renewal', density=True)
axes[1].hist(proba[y_test == 1], bins=60, alpha=0.6, color='#F44336',
             label='Churn', density=True)
axes[1].axvline(best_result['threshold'], color='black', linestyle='--',
                label=f'Umbral óptimo = {best_result["threshold"]:.2f}')
axes[1].set_title('Distribución de scores de probabilidad')
axes[1].set_xlabel('P(churn)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Cargar modelo guardado

In [ ]:
artifact = joblib.load(MODELS_DIR / 'best_model.joblib')
print(f"Modelo cargado: {artifact['name']}")
print(f"Features: {artifact['features']}")